# 02 — SQLite Inventory AI Agent
### AI Applications | AI Engineer Roadmap

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---


## The Capstone Portfolio Project

Full AeroAgent pipeline:
- SQLite database with parts and alternates
- LLM email extraction (structured JSON output)
- Intelligent routing with alternate parts
- AI quote generation
- Gradio UI

This is your **portfolio showpiece**.


In [1]:
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY')

In [2]:
# !pip install anthropic gradio -q
import sqlite3, json, os

DB = 'aeroagent.db'

def init_db():
    conn = sqlite3.connect(DB); conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS parts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            part_number TEXT UNIQUE, description TEXT, aircraft_type TEXT,
            condition TEXT, quantity INTEGER DEFAULT 0,
            price REAL, cert TEXT, alternate_pn TEXT
        );
    """)
    data = [
        ('AV-29341-A','Hydraulic Actuator Flap','Boeing 737-800','Serviceable',3,18750,'FAA 8130-3','AV-29341-B'),
        ('AV-29341-B','Hydraulic Actuator Alt','Boeing 737-800','Serviceable',1,17200,'EASA Form 1',None),
        ('GE-CF56-7B','CFM56-7B Fan Blade','Boeing 737 NG','Overhauled',0,94500,'FAA 8130-3','GE-CF56-7B-ALT'),
        ('GE-CF56-7B-ALT','CFM56-7B Fan Blade Exchange','Boeing 737 NG','Serviceable',2,88000,'FAA 8130-3',None),
        ('AIR-A320-BLV','Bleed Air Valve A320','Airbus A320','New',5,6200,'EASA Form 1',None),
        ('NGS-777-STR','Nose Gear Steering Actuator','Boeing 777','Serviceable',1,31400,'FAA 8130-3',None),
    ]
    for p in data: conn.execute('INSERT OR IGNORE INTO parts VALUES (NULL,?,?,?,?,?,?,?,?)',p)
    conn.commit(); conn.close()
    print(f'DB ready: {len(data)} parts loaded')

init_db()

DB ready: 6 parts loaded


In [3]:
!pip install anthropic -q
import anthropic
import os
import json

def extract_from_email(email_text):
    client = anthropic.Anthropic(api_key=api_key)
    r = client.messages.create(
        model='claude-sonnet-4-5', max_tokens=512,
        system='Extract AOG request data. Return ONLY JSON with no markdown, no backticks, no explanation: {"part_number":null,"description":null,"aircraft_type":null,"quantity":1,"airline":null,"contact":null,"urgency":"ROUTINE"}',
        messages=[{'role':'user','content':f'Extract from this email:\n{email_text}'}]
    )
    raw = r.content[0].text.strip()
    print("Raw response:", raw)  # 👈 shows exactly what Claude returned

    # Strip markdown code fences if present
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()

    return json.loads(raw)

def query_db(pn, desc=None):
    conn = sqlite3.connect(DB); conn.row_factory = sqlite3.Row
    part = conn.execute('SELECT * FROM parts WHERE UPPER(part_number)=UPPER(?)',(pn,)).fetchone()
    if not part and desc:
        part = conn.execute('SELECT * FROM parts WHERE description LIKE ?',(f'%{desc.split()[0]}%',)).fetchone()
    if not part: conn.close(); return None, None
    part = dict(part); alt = None
    if part['quantity']==0 and part['alternate_pn']:
        a = conn.execute('SELECT * FROM parts WHERE part_number=?',(part['alternate_pn'],)).fetchone()
        if a: alt = dict(a)
    conn.close(); return part, alt

def generate_quote(ext, part, alt):
    active = alt if (part['quantity']==0 and alt) else part
    is_alt = part['quantity']==0 and alt is not None
    ctx = f'ALTERNATE {active["part_number"]}' if is_alt else f'IN STOCK: {active["part_number"]}'
    client = anthropic.Anthropic(api_key=api_key)
    r = client.messages.create(
        model='claude-sonnet-4-5', max_tokens=512,
        system='You are a senior sales specialist at Eagle Jet Solutions, global AOG parts supplier.',
        messages=[{'role':'user','content':f'Write professional AOG quote. Customer: {ext.get("airline","Valued Customer")}. Urgency: {ext.get("urgency")}. Part: {ctx}, Price: ${active["price"]:,.0f}, Cert: {active["cert"]}. Under 150 words.'}]
    )
    return r.content[0].text

print('Pipeline functions ready')


Pipeline functions ready


In [4]:
def pipeline(email):
    if not email.strip(): return 'Enter email','','',''
    try:
        ext = extract_from_email(email)
        part, alt = query_db(ext.get('part_number',''), ext.get('description',''))
        if not part: return json.dumps(ext,indent=2),'Part not found in DB','Manual sourcing needed','Not Found'
        is_oos = part['quantity']==0
        inv = json.dumps({'primary':part,'alternate':alt},indent=2)
        quote = generate_quote(ext, part, alt)
        status = 'OOS + Alternate Offered' if is_oos and alt else 'OOS' if is_oos else f'In Stock ({part["quantity"]} units)'
        return json.dumps(ext,indent=2), inv, quote, status
    except Exception as e:
        import traceback
        err = traceback.format_exc()  # 👈 full error details
        print(err)                    # 👈 prints in Colab cell output
        return f'Error: {err}','','',f'Error'

In [5]:
import gradio as gr

SAMPLES = {
    'AOG 737 Actuator': 'URGENT AOG at JFK. Need P/N AV-29341-A hydraulic actuator. Boeing 737-800. 1 unit. FAA docs. Aircraft grounded. - Mike Chen, Pacific Air',
    'OOS Engine Part':  'Emergency: GE-CF56-7B fan blade. Boeing 737 NG. Overhauled. Aircraft on ground ATL. - Sarah Johnson, Delta Ops',
    'Routine A320':     'Please quote AIR-A320-BLV x2 units. A320. New, EASA docs. Not AOG. Klaus Muller, EuroJet',
}


with gr.Blocks(theme=gr.themes.Soft(), title='AeroAgent') as demo:
    gr.Markdown('# AeroAgent - Autonomous AOG Quoting Engine\nAI-powered aviation parts pipeline')
    with gr.Row():
        with gr.Column():
            sample = gr.Dropdown(list(SAMPLES.keys()), label='Load Sample Email')
            email_in = gr.Textbox(label='Email Input', lines=6)
            btn = gr.Button('Run Pipeline', variant='primary')
            status = gr.Textbox(label='Status')
        with gr.Column():
            extracted = gr.Code(label='Extracted JSON', language='json')
            inventory = gr.Code(label='Inventory Result', language='json')
    quote = gr.Textbox(label='Generated Quote Email', lines=10)
    sample.change(lambda s: SAMPLES.get(s,''), sample, email_in)
    btn.click(pipeline, email_in, [extracted, inventory, quote, status])

demo.launch(share=True)

/tmp/ipykernel_14648/321982600.py:10: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='AeroAgent') as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7bc60d718f33cedcb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
test_email = "URGENT AOG at JFK. Need P/N AV-29341-A hydraulic actuator. Boeing 737-800. 1 unit. FAA docs. Aircraft grounded. - Mike Chen, Pacific Air"

ext = extract_from_email(test_email)
print("Extracted:", ext)

part, alt = query_db(ext.get('part_number',''), ext.get('description',''))
print("Part:", part)
print("Alt:", alt)

Raw response: ```json
{
  "part_number": "AV-29341-A",
  "description": "hydraulic actuator",
  "aircraft_type": "Boeing 737-800",
  "quantity": 1,
  "airline": "Pacific Air",
  "contact": "Mike Chen",
  "urgency": "AOG"
}
```
Extracted: {'part_number': 'AV-29341-A', 'description': 'hydraulic actuator', 'aircraft_type': 'Boeing 737-800', 'quantity': 1, 'airline': 'Pacific Air', 'contact': 'Mike Chen', 'urgency': 'AOG'}
Part: {'id': 1, 'part_number': 'AV-29341-A', 'description': 'Hydraulic Actuator Flap', 'aircraft_type': 'Boeing 737-800', 'condition': 'Serviceable', 'quantity': 3, 'price': 18750.0, 'cert': 'FAA 8130-3', 'alternate_pn': 'AV-29341-B'}
Alt: None
